# LLM 4일차 (2) — PDF 기반 RAG : 대한민국 헌법 Q&A 챗봇

수업 필기 `RAG_실습08`(내 필기)과 강의본 `RAG_실습8_1` 을 정리 · 개선한 노트입니다. (교재 `AI Agent` **p.44** — 실습 : PDF 기반 RAG(헌법 Q&A 챗봇) 구현)

```
PyMuPDFLoader(PDF) → 페이지 14개 → 분할 → 임베딩 → Chroma → 검색(k=2) → 프롬프트(조항 번호를 밝혀라) → LLM
```

- 매뉴얼 챗봇 · 벡터 저장소 · FAISS vs Chroma · 메타데이터 필터는 [`rag_vectorstore_chroma.ipynb`](./rag_vectorstore_chroma.ipynb)
- 핵심 개선 : 글자 수(500자)로 자르면 **조문이 중간에서 잘린다** → **조문(제N조) 하나 = 청크 하나** + 메타데이터(조 번호 · 장)

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다(Day 046 ~ 048 과 같은 방식).
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩 모델**(`jhgan/ko-sroberta-multitask`, Day 041 · 045 에서 쓴 것) 또는 **가짜 LLM** 으로 실제로 돌린 셀 — 검색 순위 · 점수는 수업의 `text-embedding-3-small` 과 다를 수 있다 |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 (불러오기 · 분할 · DB 파일 들여다보기 · 수업 기록 분석) |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `pymupdf` · `langchain-community` | `PyMuPDFLoader` (PDF → 페이지별 `Document`) | `pip install pymupdf langchain-community` |
| `langchain-text-splitters` · `langchain-chroma` | 분할 · 벡터 저장소 | `pip install langchain-text-splitters langchain-chroma` |
| `langchain-openai` | `OpenAIEmbeddings` · `ChatOpenAI` (⏳) | `pip install langchain-openai` |
| `langchain-huggingface` · `sentence-transformers` | 🧪 로컬 한국어 임베딩 | `pip install langchain-huggingface sentence-transformers` |
| `python-dotenv` · `pandas` | API 키 · 표 | `pip install python-dotenv pandas` |

**데이터** : 노트북과 같은 폴더의 `대한민국헌법(헌법)(제00010호)(19880225).pdf` (국가법령정보센터, 14쪽 — 법령은 저작권 보호 대상이 아니다). 수업 기록 분석(§6)은 같은 폴더의 필기 원본 `RAG_실습8_1.ipynb` 를 읽는다.
**실행** : 맨 위부터 **모두 실행**. 🧪 로컬 임베딩은 처음 한 번 모델(약 440MB)을 받는다. 노트북 폴더에 `chroma_db_constitution_*` 폴더가 생긴다.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q pymupdf langchain-community langchain-text-splitters langchain-chroma langchain-openai langchain-huggingface sentence-transformers python-dotenv pandas

In [2]:
import os, re, json, shutil, warnings
import pandas as pd
warnings.filterwarnings('ignore', message='.*sunset.*')
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

for d in ['chroma_db_constitution_500', 'chroma_db_constitution_article']:
    shutil.rmtree(d, ignore_errors=True)           # 연습용 DB 는 매번 새로

In [3]:
# .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
from dotenv import load_dotenv, find_dotenv
loaded = load_dotenv(find_dotenv(usecwd=True))      # 지금 폴더에서 위로 올라가며 .env 를 찾는다
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

.env 읽음 : False | OPENAI_API_KEY 있음 : False


## 1. PDF 불러오기 — `PyMuPDFLoader`

In [4]:
pdf_file_path = '대한민국헌법(헌법)(제00010호)(19880225).pdf'
pages = PyMuPDFLoader(file_path=pdf_file_path).load()          # 페이지 하나 = Document 하나
print('Document(페이지) 수 :', len(pages), '| 전체 글자 수 :', sum(len(p.page_content) for p in pages))
print('metadata 키', len(pages[0].metadata), '개 :', sorted(pages[0].metadata))
print('\n5쪽 앞부분 :', repr(pages[4].page_content[:140]))
print("pages[4].metadata['page'] =", pages[4].metadata['page'], '← 0부터 센다 (PDF 의 5쪽)')

Document(페이지) 수 : 14 | 전체 글자 수 : 20881
metadata 키 16 개 : ['author', 'creationDate', 'creationdate', 'creator', 'file_path', 'format', 'keywords', 'modDate', 'moddate', 'page', 'producer', 'source', 'subject', 'title', 'total_pages', 'trapped']

5쪽 앞부분 : '법제처                                                            5                                                       국가법령정보센터\n대한민국헌법\n③국회의원'
pages[4].metadata['page'] = 4 ← 0부터 센다 (PDF 의 5쪽)


- `metadata['page']` 는 **0부터** 센다 — 수업 출력에서 `'page': 4` 인 청크의 머리글이 `법제처 5` 인 이유. 출처를 보여 줄 때는 `+1`
- 모든 페이지 첫 줄에 머리글 `법제처 N 국가법령정보센터` · `대한민국헌법` 이 붙어 있다 → 청크에 섞여 들어가 **검색과 상관없는 글자**가 된다
- PDF 의 줄바꿈은 **화면 폭에서 끊긴 것**이라 단어 중간에서도 끊긴다(`민족의 단\n결을` · `확고\n히`)

## 2. 수업 방식 — 500자로 자르기

In [5]:
text_splitter = RecursiveCharacterTextSplitter(separators=['\n\n', '\n', ' ', ''], chunk_size=500, chunk_overlap=0)
docs500 = text_splitter.split_documents(pages)
lengths = [len(d.page_content) for d in docs500]
HEADER = re.compile(r'법제처\s+\d+\s+국가법령정보센터\n대한민국헌법\n')
begins_ok = lambda t: re.match(r'\s*(제\d+조|제\d+장|부칙|전문|대한민국헌법|\s)', t)     # 조문 · 장 머리에서 시작하면 OK
page_first = [d for d in docs500 if '국가법령정보센터' in d.page_content]                  # 페이지의 첫 청크 (머리글로 시작)
cut_by_size = [d for d in docs500 if d not in page_first and not begins_ok(d.page_content)]
cut_by_page = [d for d in page_first if not begins_ok(HEADER.sub('', d.page_content))]
print('청크', len(docs500), '개 · 길이', min(lengths), '~', max(lengths), '자 (수업 출력도 52개)')
print('머리글이 섞인 청크 :', len(page_first), '개 (페이지마다 첫 청크)')
print(f'조문 중간에서 시작하는 청크 : {len(cut_by_size) + len(cut_by_page)}개 = 500자 한도에서 잘림 {len(cut_by_size)} + 페이지가 바뀌어 잘림 {len(cut_by_page)}')
print('  페이지에서 잘린 예 :', [HEADER.sub('', d.page_content)[:14] for d in cut_by_page[:4]])
c65 = next(d for d in docs500 if '제65조 ' in d.page_content)
print(f"\n제65조(탄핵소추)가 든 청크 (page {c65.metadata['page']}) 의 끝 :", repr(c65.page_content[-45:]))
nxt = docs500[docs500.index(c65) + 1]
print(f"다음 청크 (page {nxt.metadata['page']}) 의 시작 :", repr(HEADER.sub('', nxt.page_content)[:30]))

청크 52 개 · 길이 65 ~ 499 자 (수업 출력도 52개)
머리글이 섞인 청크 : 14 개 (페이지마다 첫 청크)
조문 중간에서 시작하는 청크 : 27개 = 500자 한도에서 잘림 20 + 페이지가 바뀌어 잘림 7
  페이지에서 잘린 예 : ['③국회의원의 선거구와 비례', '②정부는 회계연도마다 예산', '있어야 한다.\n③탄핵소추의', '②대통령은 국가의 안위에 ']

제65조(탄핵소추)가 든 청크 (page 5) 의 끝 : '대한 탄핵소추는 국회재적의원 과반수의 발의와 국회재적의원 3분의 2 이상의 찬성이'
다음 청크 (page 6) 의 시작 : '있어야 한다.\n③탄핵소추의 의결을 받은 자는 탄핵심판이'


- `chunk_size=500` 은 글자 수만 본다. 조문 경계를 모르니 **52개 중 27개**가 조문 중간에서 시작한다
- 그중 7개는 **페이지 경계** 때문이다. `split_documents()` 는 페이지 `Document` 를 **하나씩 따로** 자르므로 청크가 페이지를 넘지 못한다 — 제65조 ② '대통령 탄핵소추는 … 3분의 2 이상의 찬성이' 는 PDF 6쪽(`page` 5) 끝, '**있어야 한다**' 는 7쪽(`page` 6) 머리글 뒤로 갈렸다(수업 출력 §6 에도 그대로). `chunk_size` 를 키워도 이 7개는 그대로다
- 헌법은 **조문**이라는 뚜렷한 단위가 있다 → 조문 하나를 청크 하나로

## 3. 개선 — 조문(제N조) 하나 = 청크 하나 + 메타데이터

In [6]:
HEADER = re.compile(r'법제처\s+\d+\s+국가법령정보센터\n대한민국헌법\n')

def article_documents(pages):
    '''머리글을 지우고 페이지를 이은 뒤, 조문 하나를 Document 하나로. metadata = 조 번호 · 장 · 본문/부칙'''
    text = '\n'.join(HEADER.sub('', p.page_content) for p in pages)       # 페이지 끝에 줄바꿈이 없어 '\n' 으로 잇는다
    body, _, addenda = text.partition('부칙 <제10호,1987. 10. 29.>')
    preamble = body.split('제1장 총강')[0]
    docs = [Document('전문\n' + preamble[preamble.find('유구한'):].strip(), metadata={'article': 0, 'chapter': '전문', 'part': '본문'})]
    chapter = '전문'
    for piece in re.split(r'\n\s*(제\d+장 [^\n]+|제\d+[절관] [^\n]+)\s*\n', '\n' + body[body.find('제1장 총강'):]):
        piece = piece.strip()
        if re.fullmatch(r'제\d+장 .+', piece):
            chapter = piece; continue
        if re.fullmatch(r'제\d+[절관] .+', piece):                         # 제4장의 절 · 관 제목은 건너뜀
            continue
        for m in re.finditer(r'(?ms)^제(\d+)조 .*?(?=^제\d+조 |\Z)', piece):
            docs.append(Document(m.group(0).replace('\n \n', '\n').strip(),
                                 metadata={'article': int(m.group(1)), 'chapter': chapter, 'part': '본문'}))
    for m in re.finditer(r'(?ms)^제(\d+)조 .*?(?=^제\d+조 |\Z)', addenda.strip()):
        docs.append(Document('부칙 ' + m.group(0).strip(), metadata={'article': int(m.group(1)), 'chapter': '부칙', 'part': '부칙'}))
    return docs

articles = article_documents(pages)
body_nums = [d.metadata['article'] for d in articles if d.metadata['part'] == '본문' and d.metadata['article'] > 0]
print('Document', len(articles), '개 = 전문 1 + 본문 조문', len(body_nums), '+ 부칙', len(articles) - 1 - len(body_nums))
print('제1조 ~ 제130조 빠짐 없음 :', body_nums == list(range(1, 131)))
print('머리글 남은 청크 :', sum('국가법령정보센터' in d.page_content for d in articles))
lens = sorted(len(d.page_content) for d in articles)
print('길이 :', lens[0], '~', lens[-1], '자 (가운데', lens[len(lens) // 2], ') · 가장 긴 조문 : 제12조(신체의 자유)')
pd.Series([d.metadata['chapter'] for d in articles]).value_counts().to_frame('조문 수').T

Document 137 개 = 전문 1 + 본문 조문 130 + 부칙 6
제1조 ~ 제130조 빠짐 없음 : True
머리글 남은 청크 : 0
길이 : 18 ~ 755 자 (가운데 102 ) · 가장 긴 조문 : 제12조(신체의 자유)


,제4장 정부,제2장 국민의 권리와 의무,제3장 국회,제5장 법원,제1장 총강,제9장 경제,부칙,제6장 헌법재판소,제7장 선거관리,제10장 헌법개정,제8장 지방자치,전문
조문 수,35,30,26,10,9,9,6,3,3,3,2,1


In [7]:
for n in [42, 65]:
    d = next(a for a in articles if a.metadata['article'] == n and a.metadata['part'] == '본문')
    print(d.metadata); print(d.page_content[:230], '\n')

{'article': 42, 'chapter': '제3장 국회', 'part': '본문'}
제42조 국회의원의 임기는 4년으로 한다. 

{'article': 65, 'chapter': '제3장 국회', 'part': '본문'}
제65조 ①대통령ㆍ국무총리ㆍ국무위원ㆍ행정각부의 장ㆍ헌법재판소 재판관ㆍ법관ㆍ중앙선거관리위원회 위원ㆍ감사
원장ㆍ감사위원 기타 법률이 정한 공무원이 그 직무집행에 있어서 헌법이나 법률을 위배한 때에는 국회는 탄핵의
소추를 의결할 수 있다.
②제1항의 탄핵소추는 국회재적의원 3분의 1 이상의 발의가 있어야 하며, 그 의결은 국회재적의원 과반수의 찬성이
있어야 한다. 다만, 대통령에 대한 탄핵소추는 국회재적의원 과반수 



- 같은 제N조가 본문과 부칙에 둘 다 있어서(부칙 제1조 ~ 제6조) `part` 로 구분하고, 부칙 청크 앞에는 `부칙` 을 붙였다
- 이제 검색된 청크의 `metadata` 로 **'제42조 · 제3장 국회'** 를 출처로 보여 줄 수 있고, '장' 으로 필터도 걸 수 있다(§7)

## 4. 🧪 검색 비교 — 500자 청크 vs 조문 청크

In [8]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
local_emb = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask',      # 처음 한 번 약 440MB 내려받음
                                  encode_kwargs={'normalize_embeddings': True})  # 길이 1 로 → 내적 = 코사인
v = local_emb.embed_query('환불 받고 싶어요')
print('벡터 길이 :', len(v), '| 한 번에 읽는 최대 토큰 :', local_emb._client.max_seq_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

벡터 길이 : 768 | 한 번에 읽는 최대 토큰 : 128


In [9]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

store500 = Chroma.from_documents(docs500, local_emb, persist_directory='chroma_db_constitution_500',
                                 collection_configuration={'hnsw': {'space': 'cosine'}})
store_art = Chroma.from_documents(articles, local_emb, persist_directory='chroma_db_constitution_article',
                                  collection_configuration={'hnsw': {'space': 'cosine'}})
tok = local_emb._client.tokenizer
over = lambda ds: sum(len(tok(d.page_content)['input_ids']) > local_emb._client.max_seq_length for d in ds)
print(f'로컬 모델이 앞부분만 읽는(128 토큰 초과) 청크 : 500자 {over(docs500)}/{len(docs500)} · 조문 {over(articles)}/{len(articles)}')

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (173 > 128). Running this sequence through the model will result in indexing errors


로컬 모델이 앞부분만 읽는(128 토큰 초과) 청크 : 500자 43/52 · 조문 20/137


In [10]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

QUESTIONS = [('국회의원의 임기는 몇 년인가?', 42), ('국무위원은 누가 임명해?', 87), ('대통령은 연임할 수 있나요?', 70),
             ('법률안은 누가 제출할 수 있어?', 52), ('언론의 자유가 보장되나요?', 21), ('대통령을 탄핵소추하려면 몇 명이 찬성해야 해?', 65),
             ('헌법재판소 재판관은 몇 명이야?', 111), ('대법원장의 임기는?', 105), ('최저임금제는 헌법에 있나요?', 32),
             ('대통령 후보가 되려면 몇 살 이상이어야 해?', 67), ('대한민국의 영토는 어디까지야?', 3), ('국회의원이 체포되지 않는 경우는?', 44),
             ('깨끗한 환경에서 살 권리가 있나요?', 35), ('감사원은 무슨 일을 해?', 97), ('헌법을 바꾸려면 국민투표가 필요해?', 130),
             ('교육을 받을 권리', 31), ('세금을 내야 할 의무가 있나요?', 38), ('군대에 가야 하는 의무', 39),
             ('지방자치단체는 무엇을 하나요?', 117), ('계엄은 언제 선포할 수 있어?', 77)]

def hit(doc, n, by_article):                       # 정답 조문이 이 청크에 들어 있나
    if by_article:
        return doc.metadata['article'] == n and doc.metadata['part'] == '본문'
    return f'제{n}조 ' in doc.page_content

rows = []
for q, n in QUESTIONS:
    r500 = store500.similarity_search(q, k=2); rart = store_art.similarity_search(q, k=2)
    rows.append({'질문': q, '정답': f'제{n}조',
                 '500자 1등': hit(r500[0], n, False), '500자 2등 안': any(hit(d, n, False) for d in r500),
                 '조문 1등': hit(rart[0], n, True), '조문 2등 안': any(hit(d, n, True) for d in rart),
                 '조문 1등 청크': f"제{rart[0].metadata['article']}조"})
table = pd.DataFrame(rows)
print(table[['500자 1등', '500자 2등 안', '조문 1등', '조문 2등 안']].sum().to_string(), f'\n(질문 {len(table)}개 중 정답 조문을 찾은 수)')
table

500자 1등      12
500자 2등 안    13
조문 1등        15
조문 2등 안      19 
(질문 20개 중 정답 조문을 찾은 수)


,질문,정답,500자 1등,500자 2등 안,조문 1등,조문 2등 안,조문 1등 청크
0,국회의원의 임기는 몇 년인가?,제42조,True,True,False,True,제41조
1,국무위원은 누가 임명해?,제87조,True,True,True,True,제87조
2,대통령은 연임할 수 있나요?,제70조,True,True,False,False,제68조
3,법률안은 누가 제출할 수 있어?,제52조,True,True,True,True,제52조
4,언론의 자유가 보장되나요?,제21조,True,True,True,True,제21조
5,대통령을 탄핵소추하려면 몇 명이 찬성해야 해?,제65조,False,False,True,True,제65조
6,헌법재판소 재판관은 몇 명이야?,제111조,False,False,True,True,제111조
7,대법원장의 임기는?,제105조,True,True,False,True,제104조
8,최저임금제는 헌법에 있나요?,제32조,True,True,True,True,제32조
9,대통령 후보가 되려면 몇 살 이상이어야 해?,제67조,False,False,False,True,제68조


- 같은 임베딩 모델 · 같은 `k=2` 인데 **청크를 조문 단위로 바꾼 것만으로** 2등 안에 정답이 드는 질문이 크게 늘었다(위 합계). 500자 청크는 조문 서너 개가 섞여 벡터 하나의 뜻이 흐려지고, 로컬 모델은 128 토큰 뒤를 아예 읽지 못한다
- OpenAI `text-embedding-3-small` 은 8,191 토큰까지 읽어서 '뒤가 잘리는' 문제는 없지만, **여러 조문이 한 벡터에 섞이는** 문제는 같다 — ⏳ 셀(§8)로 직접 비교해 보기

## 5. 데이터 밖 질문 — 점수 기준으로 거르기

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

user_query ="심각한 질병 순위"
output = retriever.invoke(user_query)
print(output)

[Document(metadata={'creator': '', 'title': '', 'file_path': '대한민국헌법(헌법)(제00010호)(19880225).pdf', 'creationDate': "D:20240401212624+09'00'", 'source': '대한민국헌법(헌법)(제00010호)(19880225).pdf', 'trapped': '', 'keywords': '', 'subject': '', 'modDate': "D:20240401212624+09'00'", 'creationdate': '2024-04-01T21:26:24+09:00', 'producer': 'iText 2.1.7 by 1T3XT', 'total_pages': 14, 'format': 'PDF 1.4', 'moddate': '2024-04-01T21:26:24+09:00', 'page': 11, 'author': ''}, page_content='법제처                                                            12                                                       국가법령정보센터\n대한민국헌법\n④위원은 정당에 가입하거나 정치에 관여할 수 없다.\n⑤위원은 탄핵 또는 금고 이상의 형의 선고에 의하지 아니하고는 파면되지 아니한다.\n⑥중앙선거관리위원회는 법령의 범위 안에서 선거관리ㆍ국민투표관리 또는 정당사무에 관한 규칙을 제정할 수 있\n으며, 법률에 저촉되지 아니하는 범위 안에서 내부규율에 관한 규칙을 제정할 수 있다.\n⑦각급 선거관리위원회의 조직ㆍ직무범위 기타 필요한 사항은 법률로 정한다.\n \n제115조 ①각급 선거관리위원회는 선거인명부의 작성 등 선거사무와 국민투표사무에 관하여 관계 행정기관에 필요한\n지시를 할 수 있다.\n②제1항의 지시를 받은 당해 행정기관은 이에 응하여야 한다.'), Document(metadata={'page': 12, 'modDate': "D

In [12]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

OUTSIDE = ['심각한 질병 순위', '오늘 점심 뭐 먹지?', '비트코인 가격 알려줘', '파이썬 리스트 정렬하는 법', '서울 날씨 어때?',
           '손흥민 소속팀은?', '김치찌개 레시피', '아이폰 최신 모델은?', '주식 투자 방법', '강아지 산책은 하루 몇 번?']
top1 = lambda q: store_art.similarity_search_with_relevance_scores(q, k=1)[0][1]
inside_scores = [top1(q) for q, _ in QUESTIONS]
outside_scores = [top1(q) for q in OUTSIDE]
print(f'헌법 질문 20개  1등 점수 : {min(inside_scores):.3f} ~ {max(inside_scores):.3f}')
print(f'헌법 밖 질문 10개 1등 점수 : {min(outside_scores):.3f} ~ {max(outside_scores):.3f}')
for th in [0.25, 0.30, 0.35, 0.40]:
    print(f'기준 {th:.2f} : 헌법 질문 통과 {sum(s >= th for s in inside_scores):2d}/20 · 헌법 밖 걸러짐 {sum(s < th for s in outside_scores):2d}/10')

헌법 질문 20개  1등 점수 : 0.450 ~ 0.726
헌법 밖 질문 10개 1등 점수 : 0.166 ~ 0.262
기준 0.25 : 헌법 질문 통과 20/20 · 헌법 밖 걸러짐  9/10
기준 0.30 : 헌법 질문 통과 20/20 · 헌법 밖 걸러짐 10/10
기준 0.35 : 헌법 질문 통과 20/20 · 헌법 밖 걸러짐 10/10
기준 0.40 : 헌법 질문 통과 20/20 · 헌법 밖 걸러짐 10/10


In [13]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

threshold_retriever = store_art.as_retriever(search_type='similarity_score_threshold',
                                             search_kwargs={'score_threshold': 0.30, 'k': 2})
for q in ['심각한 질병 순위', '국회의원의 임기는 몇 년인가?']:
    found = threshold_retriever.invoke(q)
    print(f'{q:18s} → {len(found)}개', [f"제{d.metadata['article']}조" for d in found])

No relevant docs were retrieved using the relevance score threshold 0.3


심각한 질병 순위          → 0개 []
국회의원의 임기는 몇 년인가?   → 2개 ['제41조', '제42조']


- 수업 기록 : 헌법과 상관없는 '심각한 질병 순위' 에도 검색 도구는 **중앙선거관리위원회 · 헌법개정** 조문 2개를 돌려줬다 — 검색 도구는 '가장 덜 먼' k개를 고를 뿐, **관계가 있는지는 모른다**
- 점수 기준을 두면 걸러진 질문은 LLM 을 부르지 않고 바로 '찾을 수 없습니다' 로 답할 수 있다(요금 · 환각 둘 다 줄어듦). 기준값은 위처럼 **안 / 밖 질문을 직접 재서** 정한다 — 모델을 바꾸면 다시 잰다

## 6. 수업 실행 기록 분석 (강의본 `RAG_실습8_1`)

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

user_query = "국회의원의 임기는 몇 년인가?"
output = retriever.invoke(user_query)
print(f"검색 결과: \n{output}")

검색 결과: 
[Document(id='c1109e8c-7c13-4630-b8ce-12882afb1015', metadata={'trapped': '', 'title': '', 'file_path': '대한민국헌법(헌법)(제00010호)(19880225).pdf', 'keywords': '', 'creator': '', 'subject': '', 'source': '대한민국헌법(헌법)(제00010호)(19880225).pdf', 'creationdate': '2024-04-01T21:26:24+09:00', 'creationDate': "D:20240401212624+09'00'", 'modDate': "D:20240401212624+09'00'", 'format': 'PDF 1.4', 'producer': 'iText 2.1.7 by 1T3XT', 'author': '', 'total_pages': 14, 'page': 4, 'moddate': '2024-04-01T21:26:24+09:00'}, page_content='③국회의원은 그 지위를 남용하여 국가ㆍ공공단체 또는 기업체와의 계약이나 그 처분에 의하여 재산상의 권리ㆍ이\n익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다.\n \n제47조 ①국회의 정기회는 법률이 정하는 바에 의하여 매년 1회 집회되며, 국회의 임시회는 대통령 또는 국회재적의원\n4분의 1 이상의 요구에 의하여 집회된다.\n②정기회의 회기는 100일을, 임시회의 회기는 30일을 초과할 수 없다.\n③대통령이 임시회의 집회를 요구할 때에는 기간과 집회요구의 이유를 명시하여야 한다.\n \n제48조 국회는 의장 1인과 부의장 2인을 선출한다.\n \n제49조 국회는 헌법 또는 법률에 특별한 규정이 없는 한 재적의원 과반수의 출석과 출석의원 과반수의 찬성으로 의결\n한다. 가부동수인 때에는 부결된 것으로 본다.\n \n제50조 ①국회의 회의는 공개한다. 다만, 출석의원 과반수의 찬성이 있거나 의장이 국가의 안전보장을 위하여

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

# 챗봇 while 반복문 (질문 4번 · input() 으로 넣은 질문 글자는 기록에 남지 않음)

안녕하세요! 대한민국헌법 Q&A 챗봇입니다. 헌법 조문이나 기본권 등에 대해 질문해주세요. (종료하려면 'exit' 입력)
--------------------------------------------------------------------------------
[검색된 참조 조문 근거]:
③국회의원은 그 지위를 남용하여 국가ㆍ공공단체 또는 기업체와의 계약이나 그 처분에 의하여 재산상의 권리ㆍ이
익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다.
 
제47조 ①국회의 정기회는 법률이 정하는 바에 의하여 매년 1회 집회되며, 국회의 임시회는 대통령 또는 국회재적의원
4분의 1 이상의 요구에 의하여 집회된다.
②정기회의 회기는 100일을, 임시회의 회기는 30일을 초과할 수 없다.
③대통령이 임시회의 집회를 요구할 때에는 기간과 집회요구의 이유를 명시하여야 한다.
 
제48조 국회는 의장 1인과 부의장 2인을 선출한다.
 
제49조 국회는 헌법 또는 법률에 특별한 규정이 없는 한 재적의원 과반수의 출석과 출석의원 과반수의 찬성으로 의결
한다. 가부동수인 때에는 부결된 것으로 본다.
 
제50조 ①국회의 회의는 공개한다. 다만, 출석의원 과반수의 찬성이 있거나 의장이 국가의 안전보장을 위하여 필요하다
고 인정할 때에는 공개하지 아니할 수 있다.

법제처                                                            5                                                       국가법령정보센터
대한민국헌법
③국회의원의 선거구와 비례대표제 기타 선거에 관한 사항은 법률로 정한다.
 
제42조 국회의원의 임기는 4년으로 한다.
 
제43조 국회의원은 법률이 정하는 직을 겸할 수 없다.
 
제44조 ①국회의원은 현행범인인 경우를 제외하고는 회기 중 국회의 동의없이 체포 또는 구금되지 아니한다.
②국회의원이 회기 전에 체포 또는 구금된 때에는 현행범인이 아닌 한 국회의 요

In [16]:
# 수업 기록을 다시 읽어 : 질문마다 검색된 청크가 수업 분할(500자) 몇 번째였고, 정답 조문이 들어 있었나
log = ''.join(json.load(open('RAG_실습8_1.ipynb'))['cells'][24]['outputs'][0]['text'])
turns = re.findall(r'\[검색된 참조 조문 근거\]:\n(.*?)\n-{80}\nchatbot 응답:\n(.*?)\n-{80}', log, flags=re.S)
def chunk_no(text):                                    # 수업과 같은 분할(docs500)에서 위치 찾기
    head = text.strip()[:150]                         # 머리글 청크는 앞 40자가 모두 같아서 길게 비교
    return next((i for i, d in enumerate(docs500) if d.page_content.strip().startswith(head)), None)
for t, (context, answer) in enumerate(turns, 1):
    nums = [chunk_no(part) for part in context.split('\n\n')]          # 검색된 청크 2개는 '\n\n' 으로 이어져 있다
    arts = re.findall(r'제(\d+)조', answer)
    print(f'질문 {t} : 검색 청크 {nums} (1등, 2등) · 답이 인용한 조문 {arts or "없음"}')
    print('   답 :', answer.strip()[:90])

질문 1 : 검색 청크 [17, 16] (1등, 2등) · 답이 인용한 조문 ['42']
   답 : 국회의원의 임기는 4년으로 정해져 있습니다. (제42조)
질문 2 : 검색 청크 [31, 23] (1등, 2등) · 답이 인용한 조문 ['87']
   답 : 제87조 제1항에 의하면 국무위원은 국무총리의 제청으로 대통령이 임명합니다.
질문 3 : 검색 청크 [26, 28] (1등, 2등) · 답이 인용한 조문 ['70']
   답 : 아니요. 제70조에 따르면 대통령의 임기는 5년으로 하며, 중임할 수 없습니다. 즉 한 차례의 임기만 가능하고 재선은 허용되지 않습니다.
질문 4 : 검색 청크 [18, 39] (1등, 2등) · 답이 인용한 조문 없음
   답 : 제시된 대한민국 헌법 문서에서 해당 내용을 찾을 수 없습니다.


- 질문 1(국회의원 임기) : 1등 청크는 제46조 ③ ~ 제50조로 **답이 없고**, 2등 청크에 제42조가 있었다 — `k=1` 이었으면 '찾을 수 없습니다' 가 됐을 것. 2등 청크 앞에는 머리글 `법제처 5 국가법령정보센터` 가 붙어 있다
- 질문 2 · 3(국무위원 임명 · 대통령 임기/중임) : 1등 청크에 정답 조문(제87조 · 제70조)이 있었고, 답은 규칙 1대로 **조 번호**를 밝혔다. 질문 2 의 2등 청크는 제64조 ② ~ 제65조 ② 앞부분 — 제65조 ②가 '3분의 2 이상의 찬성이' 에서 **잘려 있다**
- 질문 4 : 법률안(제51 ~ 53조) · 법원(제107 ~ 109조) 청크가 검색됐는데 답은 '찾을 수 없습니다'. 질문 글자가 기록에 없어 무엇을 물었는지 확인할 수 없다 — 질문을 함께 출력해야 기록이 쓸모 있다
- 질문 3 의 답 '중임할 수 없습니다. 즉 한 차례의 임기만 가능' 은 제70조와 맞다

## 7. 메타데이터 필터 — 장(章)으로 좁히기 (교재 p.51 응용)

In [17]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

q = '임기는 몇 년이야?'                                     # 누구의 임기인지 없는 질문
show = lambda ds: [f"제{d.metadata['article']}조({d.metadata['chapter']})" for d in ds]
print('필터 없음 :', show(store_art.similarity_search(q, k=3)))
answer_of = {'제3장 국회': 42, '제4장 정부': 70, '제5장 법원': 105, '제6장 헌법재판소': 112}   # 장마다 '임기' 조문
for ch, n in answer_of.items():
    found = store_art.similarity_search(q, k=2, filter={'chapter': ch})
    ok = any(d.metadata['article'] == n for d in found)
    names = [f"제{d.metadata['article']}조" for d in found]
    print(f"chapter = {ch:9s}: {names}  정답 제{n}조 {'✓' if ok else '✗'}")
print('$and 본문 · 제40조 이상 50 이하 :', show(store_art.similarity_search('의원의 권리', k=3, filter={
    '$and': [{'part': {'$eq': '본문'}}, {'article': {'$gte': 40}}, {'article': {'$lte': 50}}]})))

필터 없음 : ['제105조(제5장 법원)', '제68조(제4장 정부)', '제55조(제3장 국회)']


chapter = 제3장 국회   : ['제55조', '제47조']  정답 제42조 ✗
chapter = 제4장 정부   : ['제68조', '제98조']  정답 제70조 ✗


chapter = 제5장 법원   : ['제105조', '제106조']  정답 제105조 ✓
chapter = 제6장 헌법재판소: ['제112조', '제113조']  정답 제112조 ✓


$and 본문 · 제40조 이상 50 이하 : ['제40조(제3장 국회)', '제46조(제3장 국회)', '제45조(제3장 국회)']


- '임기' 처럼 여러 장에 나오는 낱말은 필터 없이 찾으면 섞인다. 화면에서 '국회' · '정부' 같은 분류를 고르게 하고 그 값을 `filter` 로 넘기면 **원하는 장 안에서만** 찾는다
- 필터가 보장하는 것은 '어느 장에서 찾나' 까지다. 그 안의 순위는 임베딩 모델 몫 — 이 로컬 모델은 법원 · 헌법재판소에서는 임기 조문을 찾았지만, 국회(제42조)와 정부(제70조)에서는 2등 안에 넣지 못했다(위 ✗). 짧은 조문 '제42조 국회의원의 임기는 4년으로 한다.' 보다 '회계연도' · '선거' 같은 낱말이 많은 조문이 앞섰다

## 8. RAG 체인 — 출처(조 번호)를 함께

In [18]:
system_prompt = '''
당신은 신뢰할 수 있는 대한민국 헌법 전문 법률 Q&A 도우미입니다.
주어진 헌법 컨텍스트 정보를 바탕으로 사용자의 질문에 정확하고 공손하게 답변해주세요.

규칙:
1. 답변 시 관련된 헌법 조항(예: 제O조 제O항)이 [컨텍스트]에 명시되어 있다면 반드시 조항 번호를 언급해주세요.
2. [컨텍스트]의 내용에 근거해서만 답변해야 하며, 자의적인 해석이나 허위 정보를 만들지 마세요.
3. [컨텍스트]에 질문과 관련된 조문 내용이 전혀 없다면, "제시된 대한민국 헌법 문서에서 해당 내용을 찾을 수 없습니다."라고 답변해주세요.

[컨텍스트]
{context}
'''
prompt = ChatPromptTemplate.from_messages([('system', system_prompt), ('human', '{question}')])
NOT_FOUND = '제시된 대한민국 헌법 문서에서 해당 내용을 찾을 수 없습니다.'

def format_articles(docs):                   # 청크 앞에 [제N조 · 장] 꼬리표 → LLM 도 조 번호를 확실히 안다
    return '\n\n'.join(f"[{d.metadata['chapter']}] {d.page_content}" for d in docs)

def ask(question, chain, retriever):
    docs = retriever.invoke(question)
    if not docs:                                              # 점수 기준을 넘는 조문이 없으면 LLM 을 부르지 않는다
        return NOT_FOUND, []
    answer = chain.invoke({'context': format_articles(docs), 'question': question})
    return answer, [f"제{d.metadata['article']}조" + ('(부칙)' if d.metadata['part'] == '부칙' else '') for d in docs]

In [19]:
# 🧪 LLM 대신 '받은 컨텍스트의 첫 줄을 되돌려 주는' 가짜 LLM — 답 내용이 아니라 체인에 무엇이 들어가는지 확인용

from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage

def _fake_llm(prompt_value):
    system = prompt_value.to_messages()[0].content
    context = system.split('[컨텍스트]')[-1].strip()
    first = context.splitlines()[0] if context else '(컨텍스트 없음)'
    return AIMessage(f'(모의 답변 · 컨텍스트 {len(context)}자를 받음) {first[:70]}')

fake_llm = RunnableLambda(_fake_llm)

In [20]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다
#   + LLM 대신 '받은 컨텍스트의 첫 줄을 되돌려 주는' 가짜 LLM — 답 내용이 아니라 체인에 무엇이 들어가는지 확인용

fake_chain = prompt | fake_llm | StrOutputParser()
for q in ['국회의원의 임기는 몇 년인가?', '대통령은 연임할 수 있나요?', '심각한 질병 순위']:
    answer, sources = ask(q, fake_chain, threshold_retriever)
    print(f'질문 : {q}\n  답 : {answer}\n  출처 : {sources or "없음"}')

질문 : 국회의원의 임기는 몇 년인가?
  답 : (모의 답변 · 컨텍스트 166자를 받음) [제3장 국회] 제41조 ①국회는 국민의 보통ㆍ평등ㆍ직접ㆍ비밀선거에 의하여 선출된 국회의원으로 구성한다.
  출처 : ['제41조', '제42조']


No relevant docs were retrieved using the relevance score threshold 0.3


질문 : 대통령은 연임할 수 있나요?
  답 : (모의 답변 · 컨텍스트 302자를 받음) [제4장 정부] 제68조 ①대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일 전에 후임자를 선거한다.
  출처 : ['제68조', '제105조']
질문 : 심각한 질병 순위
  답 : 제시된 대한민국 헌법 문서에서 해당 내용을 찾을 수 없습니다.
  출처 : 없음


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

openai_store = Chroma.from_documents(articles, OpenAIEmbeddings(model='text-embedding-3-small'),
                                     persist_directory='chroma_db_constitution_article_openai',
                                     collection_configuration={'hnsw': {'space': 'cosine'}})
rag_chain = prompt | ChatOpenAI(model='gpt-5-nano') | StrOutputParser()
retriever = openai_store.as_retriever(search_kwargs={'k': 2})   # 점수 기준은 OpenAI 임베딩으로 다시 재서 정한다
for q, _ in QUESTIONS[:5] + [('심각한 질병 순위', None)]:
    answer, sources = ask(q, rag_chain, retriever)
    print(f'질문 : {q}\n  답 : {answer}\n  출처 : {sources}\n')

# 직접 입력 (종료 : exit)
# while (q := input('질문: ').strip()).lower() != 'exit':
#     print(*ask(q, rag_chain, retriever), sep='\n  출처 : ')

- 출처를 함께 찍으면 **검색 실패가 바로 보인다** — '대통령은 연임할 수 있나요?' 의 출처가 제68조 · 제105조라 정답 제70조가 없다(이 로컬 모델의 한계, §4 표의 ✗ 하나). 이 상태로 LLM 을 부르면 규칙 3 대로 '찾을 수 없습니다' 가 나와야 맞다
- '심각한 질병 순위' 는 점수 기준(0.30)에 걸려 LLM 을 부르지 않았다

> **📝 고친 부분 (필기 `RAG_실습08` · 강의본 `RAG_실습8_1`)**
> - `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)` 를 페이지마다 → 52개 중 27개가 조문 중간에서 시작(500자 한도 20 · 페이지 경계 7) · 머리글 14번 섞임 → **머리글 제거 + 조문 단위 Document**(137개) + `metadata`(조 · 장 · 본문/부칙)
> - 검색 도구는 관계없는 질문에도 k개를 돌려준다(📼 '심각한 질병 순위') → `similarity_score_threshold` 로 거르고, 걸러지면 LLM 없이 '찾을 수 없습니다'
> - 답과 함께 **출처 조문**을 메타데이터에서 꺼내 보여 준다(교재 p.37 '출처 불분명' 에 대한 답)
> - 필기 `print("분할된 문서:\n", docs)` → 52개 전체(약 4만 7천 자) 출력 → 개수 · 길이 · 예시만
> - 필기 `if chroma_path == None: print("테스트")` → 문자열이 `None` 일 수 없어 실행되지 않는 줄 · 삭제
> - 필기 `from langchain_community.vectorstores import Chroma` → 지원 종료 예정(실습07 출력의 `LangChainDeprecationWarning`) → `from langchain_chroma import Chroma` (강의본은 이미 이것)
> - 필기 · 강의본의 `while True: try: … except Exception as e: print(…)` → 오류가 나도 **반복을 계속**해서, `input()` 자체가 실패하는 환경(노트북 자동 실행 등)에서는 오류 문구를 끝없이 찍는다 → `ask()` 함수로 분리, 반복문은 입력 확인을 먼저
> - 필기 제목 `헌번` · `백터` · `반 리스트` · `두개에 다큐멘터` → 헌법 · 벡터 · 빈 리스트 · 두 개의 Document / 필기 `.env` 절대 경로 `C:\Users\…` → `find_dotenv()`
> - 필기 · 강의본 모두 `chroma_db_constitution` 이 있으면 불러오기만 함(필기 출력 `__is not new___`) → 분할을 바꾸면 **폴더 이름도 바꾼다**(매뉴얼 노트 §5)

## 정리

| 단계 | 수업 | 개선 |
|---|---|---|
| 불러오기 | `PyMuPDFLoader` → 페이지 14개 | 같음 · `page` 는 0부터 |
| 분할 | 500자 · 겹침 0 → 52개 | 머리글 제거 · **조문 단위 137개** · 메타데이터 |
| 검색 | `k=2` | `k=2` + 점수 기준 · 필요하면 `filter={'chapter': …}` |
| 답 | 조 번호를 밝히라는 규칙 | + 메타데이터로 **출처 조문** 표시 |